### 直观理解

同时维持**多个**的候选序列, 并按照概率大小淘汰筛选

## 关键参数

### 束宽度

每一步保留的候选序列数量. 

宽度为 1 等价于贪婪解码, 宽度越高理论上的效率越高, 但是成本也越高

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-0.6B'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
)
model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

### 束搜索方式演示

In [6]:
prompt = 'hello, who are you?'

messages = [
    {'role': 'user', 'content': prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
    enable_thinking=False
)

input = tokenizer(
    text,
    return_tensors = 'pt'
)

input

{'input_ids': tensor([[151644,    872,    198,  14990,     11,    879,    525,    498,     30,
         151645,    198, 151644,  77091,    198, 151667,    271, 151668,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [7]:
text

'<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [8]:
beams = [
    {
        'input_ids': input['input_ids'],
        'attention_mask': input['attention_mask'],
        'score': 0.0,
        'finished': False
    }
]
beams

[{'input_ids': tensor([[151644,    872,    198,  14990,     11,    879,    525,    498,     30,
           151645,    198, 151644,  77091,    198, 151667,    271, 151668,    271]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]),
  'score': 0.0,
  'finished': False}]

In [88]:
import torch
import torch.nn.functional as F
candidate_beams = []
for beam in beams:
    if beam['finished']:
        candidate_beams.append(beam)
        continue

    outputs = model(
        **input
    )
    output_logits = outputs.logits
    # 只考虑 batch_size 为 1 的情况
    output_logits = output_logits[0, -1, :]
    output_logits_probs = F.softmax(output_logits)
    output_logits_prob_topk, output_logits_prob_topk_indice = torch.topk(input=output_logits_probs, k=3)

    for k in range(3):
        next_token = output_logits_prob_topk_indice[k]
        next_attention_mask = torch.ones(1,1)

        candidate_beam = {
            'input_ids': torch.cat([beam['input_ids'], torch.tensor([[next_token.item()]])], dim=1),
            'attention_mask': torch.cat([beam['attention_mask'], next_attention_mask], dim=1),
            'score': beam['score'] + output_logits_prob_topk[k],
            'finished': next_token == model.config.eos_token_id
        }
        candidate_beams.append(candidate_beam)


candidate_beams_sorted = sorted(
    candidate_beams,
    key = lambda x:x['score'],
    reverse=True
)
beams = candidate_beams_sorted[:3]

/var/folders/1k/c9npdpq131sc3pgvycnb0ts80000gn/T/ipykernel_82460/3753186549.py:15: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  output_logits_probs = F.softmax(output_logits)


In [89]:
candidate_beams_sorted

[{'input_ids': tensor([[151644,    872,    198,  14990,     11,    879,    525,    498,     30,
           151645,    198, 151644,  77091,    198, 151667,    271, 151668,    271,
             9707]]),
  'attention_mask': tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
           1.]]),
  'score': tensor(0.8945, dtype=torch.bfloat16, grad_fn=<AddBackward0>),
  'finished': tensor(False)},
 {'input_ids': tensor([[151644,    872,    198,  14990,     11,    879,    525,    498,     30,
           151645,    198, 151644,  77091,    198, 151667,    271, 151668,    271,
            13048]]),
  'attention_mask': tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
           1.]]),
  'score': tensor(0.0835, dtype=torch.bfloat16, grad_fn=<AddBackward0>),
  'finished': tensor(False)},
 {'input_ids': tensor([[151644,    872,    198,  14990,     11,    879,    525,    498,     30,
           151645,    198, 151644,  77091,    198, 151667,

## 开始正式测试束搜索方法

In [126]:
import torch
import torch.nn.functional as F

def beam_search_decoding(inputs, model, max_length=100, beam_length=3, tokenizer=None):
    beams = [
        {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
            'score': 0.0,
            'finished': False
        }
    ]

    for i in range(max_length):
        print('-'*10 + f'第 {i} 轮' + '-'*10)
        candidate_beams = []
        for beam in beams:
            if beam['finished']:
                candidate_beams.append(beam)
                continue

            outputs = model(
                input_ids = beam['input_ids'],
                attention_mask = beam['attention_mask']
            )
            output_logits = outputs.logits
            # 只考虑 batch_size 为 1 的情况
            output_logits = output_logits[0, -1, :]
            output_logits_probs = F.log_softmax(output_logits, dim=-1) # 注意, 这里的取的是对数
            output_logits_prob_topk, output_logits_prob_topk_indice = torch.topk(input=output_logits_probs, k=beam_length)

            for k in range(beam_length):
                next_token = output_logits_prob_topk_indice[k]
                next_attention_mask = torch.ones(1,1)

                candidate_beam = {
                    'input_ids': torch.cat([beam['input_ids'], torch.tensor([[next_token.item()]])], dim=1),
                    'attention_mask': torch.cat([beam['attention_mask'], next_attention_mask], dim=1),
                    'score': beam['score'] + output_logits_prob_topk[k].item(),
                    'finished': next_token.item() == model.config.eos_token_id
                }
                candidate_beams.append(candidate_beam)

        if tokenizer is not None:
            print('candidate_beams: ')
            for j in candidate_beams:
                print(
                    f"score: {j['score']}  {tokenizer.decode(j['input_ids'])}"
                )
        print()

        candidate_beams_sorted = sorted(
            candidate_beams,
            key = lambda x:x['score'],
            reverse=True
        )
        beams = candidate_beams_sorted[:beam_length]

        if tokenizer is not None:
            print('preserved_beams: ')
            for j in beams:
                print(
                    f"score: {j['score']}  {tokenizer.decode(j['input_ids'])}"
                )

        # 附件的终止条件
        if all(beam['finished'] for beam in beams):
            break
    
    best_beam = beams[0]
    return best_beam['input_ids']


In [128]:
prompt = 'hello, who are you?'

messages = [
    {'role': 'user', 'content': prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors = 'pt'
)

outputs = beam_search_decoding(inputs=inputs, model=model, tokenizer=tokenizer, beam_length=2, max_length=50)

tokenizer.decode(outputs)


----------第 0 轮----------
candidate_beams: 
score: -0.11083984375  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHello']
score: -2.484375  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHi']

preserved_beams: 
score: -0.11083984375  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHello']
score: -2.484375  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHi']
----------第 1 轮----------
candidate_beams: 
score: -0.1263427734375  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHello!']
score: -4.61083984375  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHello,']
score: -3.20703125  ['<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHi there']
score: -3

["<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHello! I'm an AI assistant designed to help with various tasks and questions. If you have any questions or need assistance, feel free to ask!<|im_end|>"]

## 优缺点

### 优点

- 质量更高
- 考虑了全局因素的影响

### 缺点

- 引入了大量额外的开销
- 多样性仍然有限

## 现在有个问题, 貌似系统很难终止